# RHF / BS-UHF orbitals -> spin-orbital FCIDUMP -> LUCJ -> Local SQD Recovery Walkthrough

This notebook runs the same pipeline this repo's `sbd` package runs at scale on Fugaku/ROQUO --
end to end, on a laptop, in about a minute. No HPC, no Prefect server, no real IBM Quantum hardware.

**The workflow does not build orbitals.** Its UHF input is a *spin-orbital* FCIDUMP
(`qcsc_workflow_utility.uhf_fcidump` format, the one the SBD `-D_UHF` build reads). The orbitals
are chosen *before* the workflow by one of two converters that live next to this repository
(plain folders, not part of qcsc-prefect):

- `rhf2uhf`: RHF (spatial) FCIDUMP -> spin-orbital FCIDUMP, orbitals unchanged (h_alpha = h_beta).
- `rhf2bsuhf`: spatial FCIDUMP -> broken-symmetry UHF (atom-localized AF guess +
  stability following) -> spin-orbital FCIDUMP in the BS-UHF MO basis.

`compute_molecular_integrals_from_uhf_fcidump` then loads the file as-is (no SCF: the reference is
the lowest `na`/`nb` orbitals by index) and runs UCCSD on top for the LUCJ `t2` seed and the
initial occupancies used by configuration recovery.

The pipeline is molecule-agnostic. This notebook uses a real Fe2S2 iron-sulfur-cluster FCIDUMP,
but nothing below is iron-specific except the AF fragment spec in Step 3.

Companions:

- [`../../docs/tutorials/run_uhf_bsuhf_any_molecule.md`](../../docs/tutorials/run_uhf_bsuhf_any_molecule.md)
  -- the full tutorial, including how to scale this up on Fugaku or ROQUO.
- [`run_recover.py`](run_recover.py) and [`README.md`](README.md) in this directory.

In [ ]:
import contextlib
import io
import json
import logging
import os
import re
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S",
)
log = logging.getLogger("walkthrough")

import prefect.logging.loggers as _plg
_orig_get_run_logger = _plg.get_run_logger
def _safe_get_run_logger(*a, **kw):
    try:
        return _orig_get_run_logger(*a, **kw)
    except Exception:
        return log
_plg.get_run_logger = _safe_get_run_logger
import prefect.logging as _pl
_pl.get_run_logger = _safe_get_run_logger
from qcsc_workflow_utility import chem as _chem
_chem.get_run_logger = _safe_get_run_logger
from sbd import solver_job as _sj
_sj.get_run_logger = _safe_get_run_logger

import ffsim
from qcsc_workflow_utility.chem import compute_molecular_integrals_from_uhf_fcidump
from qcsc_workflow_utility.uhf_fcidump import aufbau_reference, read_uhf_fcidump
from qiskit_addon_sqd.configuration_recovery import recover_configurations
from qiskit_addon_sqd.counts import bit_array_to_arrays, generate_bit_array_uniform
from sbd.lucj import create_lucj_circuit, initialize_ucj_parameters
from sbd.sqd import _compute_excitation_counts, postselect_bitstrings, subsample_open_shell

# The orbital converters are plain folders next to qcsc-prefect (uhf_sqd_oo/rhf2uhf, rhf2bsuhf).
# Override with UHF_SQD_OO_DIR if they live elsewhere.
REPO_ROOT = Path.cwd().resolve().parents[1]
CONVERTER_ROOT = Path(os.environ.get("UHF_SQD_OO_DIR", REPO_ROOT.parent))
for _d in ("rhf2uhf", "rhf2bsuhf"):
    if not (CONVERTER_ROOT / _d).is_dir():
        raise FileNotFoundError(f"{CONVERTER_ROOT / _d} not found; set UHF_SQD_OO_DIR")
    sys.path.insert(0, str(CONVERTER_ROOT / _d))
import rhf2uhf
import rhf2bsuhf

t_notebook_start = time.perf_counter()
print("Setup complete: Prefect get_run_logger patched to fall back to stdlib logging.")

## Step 0 -- Get an FCIDUMP

This example uses a real Fe2S2 iron-sulfur cluster FCIDUMP because it's a genuine, publicly
available, strongly-correlated open-shell system this repo already validates against -- swap in
your own FCIDUMP here; nothing below is iron-specific.


In [ ]:
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FCIDUMP_FILE = DATA_DIR / "fe2s2.fcidump"
FCIDUMP_URL = (
    "https://raw.githubusercontent.com/zhendongli2008/"
    "Active-space-model-for-Iron-Sulfur-Clusters/main/Fe2S2_and_Fe4S4/Fe2S2/fe2s2"
)

if not FCIDUMP_FILE.exists():
    urllib.request.urlretrieve(FCIDUMP_URL, FCIDUMP_FILE)

header_line = FCIDUMP_FILE.read_text().splitlines()[0]
norb = int(re.search(r"NORB=\s*(\d+)", header_line).group(1))
nelec = int(re.search(r"NELEC=\s*(\d+)", header_line).group(1))
ms2 = int(re.search(r"MS2=\s*(-?\d+)", header_line).group(1))
assert (norb, nelec, ms2) == (20, 30, 0), (norb, nelec, ms2)
print(f"FCIDUMP: {FCIDUMP_FILE}")
print(f"Header : NORB={norb}  NELEC={nelec}  MS2={ms2}")


## Step 1 -- RHF orbitals -> spin-orbital FCIDUMP (`rhf2uhf`)

The Fe2S2 FCIDUMP is already in canonical RHF MOs. `rhf2uhf` rewrites it in the spin-orbital
format with identical alpha and beta integrals -- no orbital change at all -- and checks that the
aufbau determinant's energy equals PySCF's closed-shell energy of the input orbitals.

In [ ]:
RHF_SO = DATA_DIR / "fe2s2_rhf_uhf.fcidump"
info_rhf = rhf2uhf.convert(str(FCIDUMP_FILE), str(RHF_SO))
e_rhf = info_rhf["E_ref"]
print(f"RHF reference (aufbau of the input orbitals): {e_rhf:.6f} Ha  "
      f"(PySCF check: {info_rhf['E_ref_pyscf']:.6f})")
assert abs(e_rhf - info_rhf["E_ref_pyscf"]) < 1e-8
assert info_rhf["aufbau_problems"] == []

## Step 2 -- Load it into the workflow (`compute_molecular_integrals_from_uhf_fcidump`)

This is the function `riken_sqd_de` calls when the solver block is `method="uhf"`. It is a Prefect
`@task`; outside a flow run we call the raw function via `.fn(...)`. It runs no SCF -- the file's
orbitals *are* the reference -- and prints one `[uhf-input] ... E_ref=... E_UCCSD=...` line, which
the analysis tooling parses.

In [ ]:
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(io.StringIO()):
    ep_rhf = compute_molecular_integrals_from_uhf_fcidump.fn(str(RHF_SO))
print(next(l for l in buf.getvalue().splitlines() if l.startswith("[uhf-input]")))
print(f"norb={ep_rhf.num_orbitals}  nelec={ep_rhf.num_electrons}  unrestricted={ep_rhf.unrestricted}")
# RHF-derived input: alpha and beta integrals are identical, so the UCCSD occupancies agree up to
# the UCCSD convergence tolerance (the amplitudes are solved per spin block).
print("max |occ_a - occ_b| =", float(np.max(np.abs(ep_rhf.initial_occupancy[0] - ep_rhf.initial_occupancy[1]))))

## Step 3 -- BS-UHF orbitals (`rhf2bsuhf`)

For a formally closed-shell but strongly correlated system, plain UHF started from the RHF density
stays at the RHF solution (RHF is a stationary point of the UHF equations). `rhf2bsuhf` seeds UHF
with an atom-localized (Noodleman) spin guess and follows internal and external instabilities to
the broken-symmetry minimum, then writes the spin-orbital FCIDUMP in that MO basis.

The guess is an `AF_GROUPS` dict of orbital-index fragments. The ranges below are this repo's
Fe2S2 spec (from `algorithms/sbd/sweep/run_fe2s2_uhf_sample_roquo.sh`). They refer to *this*
FCIDUMP's orbital order; for your own molecule build your own spec.

| fragment | orbital indices | role |
| --- | --- | --- |
| `l1` | 0, 1 | ligand orbitals (closed) |
| `fe1` | 2, 3, 4, 5, 6 | Fe #1 3d block -- spin-up (`up`) |
| `s` | 7, 8, 9, 10, 11, 12 | bridging-S 3p block (closed) |
| `fe2` | 13, 14, 15, 16, 17 | Fe #2 3d block -- spin-down (`down`) |
| `l2` | 18, 19 | ligand orbitals (closed) |

Fragments not listed in `up`/`down`/`free` are closed (doubly occupied); `up`/`down` get the
`0.5 +/- 0.5*pol` alpha/beta split (`--af-pol`, default 1.0). The converged MOs are cached
(`--cache-dir`), so a rerun only redoes the integral transform.

In [ ]:
af_groups = {
    "l1": [0, 1],
    "fe1": [2, 3, 4, 5, 6],
    "s": [7, 8, 9, 10, 11, 12],
    "fe2": [13, 14, 15, 16, 17],
    "l2": [18, 19],
    "up": ["fe1"],
    "down": ["fe2"],
}
BS_SO = DATA_DIR / "fe2s2_bsuhf_uhf.fcidump"
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    info_bs = rhf2bsuhf.convert(
        str(FCIDUMP_FILE), str(BS_SO), json.dumps(af_groups), af_pol=1.0,
        cache_dir=str(DATA_DIR / "uhf_cache"),
    )
e_bsuhf = info_bs["E_UHF"]
s2_bsuhf = info_bs["spin_square"]
src = "from cache" if info_bs["from_cache"] else f"SCF {info_bs['scf_seconds']:.0f} s"
print(f"BS-UHF energy: {e_bsuhf:.6f} Ha   <S^2>={s2_bsuhf:.4f}   ({src})")
assert abs(info_bs["E_ref"] - e_bsuhf) < 1e-8, "aufbau-by-index must be the BS-UHF determinant"
print(f"Delta vs RHF: {(e_bsuhf - e_rhf) * 1000:.3f} mHa")

buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(io.StringIO()):
    ep_bsuhf = compute_molecular_integrals_from_uhf_fcidump.fn(str(BS_SO))
print(next(l for l in buf.getvalue().splitlines() if l.startswith("[uhf-input]")))

## Comparison

Reference energies of the two inputs (nothing recomputed; both come from the converters'
metadata, and each equals the `E_ref` the workflow printed when loading the file).

In [ ]:
rows = [
    ("RHF orbitals (rhf2uhf)", e_rhf, 0.0),
    ("BS-UHF orbitals (rhf2bsuhf)", e_bsuhf, s2_bsuhf),
]
header = f"{'reference':<30} {'energy (Ha)':>16} {'<S^2>':>10}"
print(header)
print("-" * len(header))
for name, e, s2 in rows:
    print(f"{name:<30} {e:>16.6f} {s2:>10.4f}")

## Step 4 -- LUCJ circuit build

Using the BS-UHF input (`ep_bsuhf`), we build LUCJ ansatz parameters from its UCCSD `t2` amplitudes via
`sbd.lucj.initialize_ucj_parameters` (the exact function `run_local_uhf.py` uses for its UHF path)
and turn them into a circuit via `sbd.lucj.create_lucj_circuit`. Both are Prefect `@task`s, so we
again call `.fn(...)`.

To get a realistic ISA (basis-gate, hardware-connectivity) circuit with no real IBM Quantum access,
we transpile against a `GenericBackendV2` fake backend -- the same fake-backend pattern this
repo's own layout tests (`tests/test_error_aware_layout.py`) use for hardware-free transpilation
checks.

In [ ]:
norb = ep_bsuhf.num_orbitals
num_elec_a, num_elec_b = ep_bsuhf.num_electrons
aa_indices = [(p, p + 1) for p in range(norb - 1)]
ab_indices = [(p, p) for p in range(0, norb, 4)]

ucj_params = initialize_ucj_parameters.fn(
    elec_props=ep_bsuhf,
    aa_indices=aa_indices,
    ab_indices=ab_indices,
    num_walkers=1,
    randomization_factor=0.2,
    n_lucj_layers=1,
)
vir_circuit = create_lucj_circuit.fn(
    ucj_parameter=ucj_params[0],
    elec_props=ep_bsuhf,
    aa_indices=aa_indices,
    ab_indices=ab_indices,
    n_lucj_layers=1,
    use_reset_mitigation=False,
)
print(f"Virtual LUCJ circuit: {vir_circuit.num_qubits} qubits, {len(vir_circuit.data)} instructions")


In [ ]:
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

backend = GenericBackendV2(num_qubits=vir_circuit.num_qubits, seed=1)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=42)
isa_circuit = pm.run(vir_circuit)
two_qubit_gate_count = sum(1 for instr in isa_circuit.data if instr.operation.num_qubits == 2)

print(f"ISA circuit: {isa_circuit.num_qubits} qubits")
print(f"Total gates: {len(isa_circuit.data)}")
print(f"Two-qubit gates: {two_qubit_gate_count}")


## Step 5 -- Local recovery pass

One small local SQD recovery step, with no real device and no simulator: `generate_bit_array_uniform`
produces deterministic pseudo-random bitstrings -- the exact mechanism behind `quantum_source=
"random"` in the shipped scripts. The rest of the pipeline is identical to the real-hardware path:
`recover_configurations` biases the raw samples toward the mean-field occupancies, `postselect_bitstrings`
keeps only the correct-Hamming-weight (particle-number-conserving) strings, and `subsample_open_shell`
dedups and caps each spin channel independently at `sqd_dim`, producing the alpha/beta CI-string
arrays the SBD solver would diagonalize next.


In [ ]:
sqd_dim = 3000
bit_array = generate_bit_array_uniform(num_samples=sqd_dim, num_bits=norb * 2, rand_seed=24)
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

bitstrings, probs = recover_configurations(
    bitstring_matrix=raw_bitstrings,
    probabilities=raw_probs,
    avg_occupancies=ep_bsuhf.initial_occupancy,
    num_elec_a=num_elec_a,
    num_elec_b=num_elec_b,
    rand_seed=np.random.default_rng(24),
)
bitstrings_post, probs_post = postselect_bitstrings.fn(
    bitstring_matrix=bitstrings,
    probabilities=probs,
    hamming_right=num_elec_a,
    hamming_left=num_elec_b,
)

carryover_a = np.zeros((0, norb), dtype=bool)
carryover_b = np.zeros((0, norb), dtype=bool)
ci_a, ci_b = subsample_open_shell.fn(
    bitstring_matrix=bitstrings_post,
    probabilities=probs_post,
    carryover_a=carryover_a,
    carryover_b=carryover_b,
    subspace_dim=sqd_dim,
    norb=norb,
    num_elec_a=num_elec_a,
    num_elec_b=num_elec_b,
    rng=np.random.default_rng(24),
)
print(f"sqd_dim={sqd_dim}  ->  {len(ci_a)} alpha strings, {len(ci_b)} beta strings"
      f"  (net CI matrix {len(ci_a)} x {len(ci_b)} = {len(ci_a) * len(ci_b)})")


## Step 6 -- Excitation-rank breakdown

Per-spin unique-determinant counts by excitation order relative to the Hartree-Fock reference
string, in the same `S=.. D=.. T=.. Q=.. ≥5=..` format the tutorial's Step 7 (and every shipped
launcher's `[diag] recovery ...` log line) uses. This compares each **unique** alpha/beta bitstring's
popcount-XOR against the HF string -- a per-spin unique-string count, not a naive per-shot joint
alpha+beta excitation count (that was a documented bug in an earlier session: it inflates high-
excitation counts and hides how saturated the low orders already are).

`sbd.sqd._compute_excitation_counts` is the corrected, shipped implementation; this cell just calls
it on `ci_a`/`ci_b` directly.


In [ ]:
counts_a = _compute_excitation_counts(ci_a, num_elec_a)
counts_b = _compute_excitation_counts(ci_b, num_elec_b)

def _fmt(counts):
    return (f"n={counts['n_total']} unique={counts['n_unique']} HF={counts['HF']} "
            f"S={counts['S']} D={counts['D']} T={counts['T']} Q={counts['Q']} "
            f"≥5={counts['high5']} max_exc={counts['max_exc']}")

print("alpha:", _fmt(counts_a))
print("beta :", _fmt(counts_b))

t_notebook_total = time.perf_counter() - t_notebook_start
print(f"\nNotebook wall-clock (from Setup cell): {t_notebook_total:.1f}s")


## How this scales

Everything above used a tiny `sqd_dim` and a single recovery step so the whole notebook runs in
seconds. A real production run differs in three ways:

1. **Sample from real hardware** instead of `generate_bit_array_uniform` (`quantum_source=
   "real-device"` in the shipped flow).
2. **Persist the pool** instead of resampling every time.
3. **Hand the persisted pool to [`run_recover.py`](run_recover.py)** with many more recovery steps
   and a much larger `sqd_dim`, sized per
   [`docs/reference/hpc_resource_sizing.md`](../../docs/reference/hpc_resource_sizing.md), on
   Fugaku or ROQUO.

See [`docs/tutorials/run_uhf_bsuhf_any_molecule.md`](../../docs/tutorials/run_uhf_bsuhf_any_molecule.md)
for the full path from here to a checkpointed, multi-node deep run.
